In [2]:
pip install numpy soundfile sounddevice matplotlib pandas pydub tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
from IPython.display import Audio 
import pandas as pd
import os
from pathlib import Path
from itertools import product
from pydub import AudioSegment
import numpy as np
from tqdm import tqdm

In [4]:
# Пути к файлам
base_dir = "cv-corpus-21.0-2025-03-14/ru"
clips_dir = os.path.join(base_dir, "clips")
validated_path = os.path.join(base_dir, "validated.tsv")

In [5]:
# Загружаем метаданные проверенных записей
df = pd.read_csv(validated_path, sep='\t')
print(f"Всего записей: {len(df)}")

Всего записей: 170004


C:\Users\DNS\AppData\Local\Temp\ipykernel_56028\1379785600.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(validated_path, sep='\t')


In [6]:
# Фильтрация записей с указанным полом
male_speakers = df[df['gender'] == 'male_masculine'].copy()
female_speakers = df[df['gender'] == 'female_feminine'].copy()

In [7]:
male_speakers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103411 entries, 6 to 161568
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   client_id        103411 non-null  object 
 1   path             103411 non-null  object 
 2   sentence_id      103411 non-null  object 
 3   sentence         103411 non-null  object 
 4   sentence_domain  1 non-null       object 
 5   up_votes         103411 non-null  int64  
 6   down_votes       103411 non-null  int64  
 7   age              103211 non-null  object 
 8   gender           103411 non-null  object 
 9   accents          14340 non-null   object 
 10  variant          0 non-null       float64
 11  locale           103411 non-null  object 
 12  segment          973 non-null     object 
dtypes: float64(1), int64(2), object(10)
memory usage: 11.0+ MB


In [8]:
male_speakers.head(1)

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
6,09d37c3e11216db461125872a718d75bf0a8f51ede4b8e...,common_voice_ru_23236368.mp3,ce58b07444348f035808352e569fb50279c2711233d168...,Для всех нас основной проблемой является расту...,NaN,2,0,twenties,male_masculine,NaN,NaN,ru,NaN


In [9]:
female_speakers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26350 entries, 35 to 141601
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   client_id        26350 non-null  object 
 1   path             26350 non-null  object 
 2   sentence_id      26350 non-null  object 
 3   sentence         26350 non-null  object 
 4   sentence_domain  1 non-null      object 
 5   up_votes         26350 non-null  int64  
 6   down_votes       26350 non-null  int64  
 7   age              26086 non-null  object 
 8   gender           26350 non-null  object 
 9   accents          354 non-null    object 
 10  variant          0 non-null      float64
 11  locale           26350 non-null  object 
 12  segment          117 non-null    object 
dtypes: float64(1), int64(2), object(10)
memory usage: 2.8+ MB


In [ ]:
# Создаем структуру результирующего датафрейма
result_columns = [
    'ID',
    'genders',
    'pattern',
    'SNR',
    'file1',        
    'file2',        
    'output_file',  
    'sentence1',
    'sentence2',
    'duration',
    'overlap_ratio'
]

result_df = pd.DataFrame(columns=result_columns)

In [11]:
# Параметры генерации
patterns = ['Частичное перекрытие', 'Параллельный диалог']
snr_levels = [-5, 0, 5]

In [12]:
def generate_combinations(patterns, snr_levels):
    """Генератор комбинаций параметров"""
    combo_id = 1
    while True:
        for gender_combo in ['М-М', 'М-Ж', 'Ж-Ж']:
            for pattern in patterns:
                for snr in snr_levels:
                    yield combo_id, gender_combo, pattern, snr
                    combo_id += 1

In [ ]:
# Функция смешивания записей
def process_audio_pair(path1, path2, pattern, snr_db, output_dir, combo_id):
    """Обрабатывает пару аудиозаписей, возвращая только имена файлов"""
    # Получаем имена файлов
    file1 = os.path.basename(path1)
    file2 = os.path.basename(path2)
    output_file = f'combo_{combo_id}.wav'
    
    # Обработка аудио 
    audio1 = AudioSegment.from_file(path1)
    audio2 = AudioSegment.from_file(path2)
    
    target_dBFS = -20
    audio1 = audio1.apply_gain(target_dBFS - audio1.dBFS)
    audio2 = audio2.apply_gain(target_dBFS - audio2.dBFS - snr_db)
    
    if pattern == 'Параллельный диалог':
        mixed = audio1.overlay(audio2)
        overlap_ratio = min(len(audio2)/len(audio1), 1.0)
    else:
        overlap_start = len(audio1) // 2
        mixed = audio1.overlay(audio2, position=overlap_start)
        actual_overlap = min(len(audio2), len(audio1) - overlap_start)
        overlap_ratio = actual_overlap / len(audio1)
    
    # Сохранение 
    os.makedirs(output_dir, exist_ok=True)
    mixed.export(os.path.join(output_dir, output_file), format='wav')
    
    return {
        'file1': file1,
        'file2': file2,
        'output_file': output_file,
        'duration': len(mixed)/1000,
        'overlap_ratio': overlap_ratio
    }

In [ ]:
def create_audio_combinations(
    male_df, female_df, base_audio_dir, output_dir,
    patterns, snr_levels, total_combinations
):
    result_data = []
    gen = generate_combinations(patterns, snr_levels)

    for _ in range(total_combinations):
        combo_id, gender_combo, pattern, snr = next(gen)
        
        try:
            # Выбор дикторов
            if gender_combo == 'М-М':
                speaker1 = male_df.sample(1).iloc[0]
                speaker2 = male_df[male_df['client_id'] != speaker1['client_id']].sample(1).iloc[0]
            elif gender_combo == 'М-Ж':
                speaker1 = male_df.sample(1).iloc[0]
                speaker2 = female_df.sample(1).iloc[0]
            else:
                speaker1 = female_df.sample(1).iloc[0]
                speaker2 = female_df[female_df['client_id'] != speaker1['client_id']].sample(1).iloc[0]

            # Полные пути для обработки
            path1 = os.path.join(base_audio_dir, speaker1['path'])
            path2 = os.path.join(base_audio_dir, speaker2['path'])
            
            if not os.path.exists(path1) or not os.path.exists(path2):
                continue

            # Обработка 
            audio_info = process_audio_pair(path1, path2, pattern, snr, output_dir, combo_id)
            
            result_data.append({
                'ID': combo_id,
                'genders': gender_combo,
                'pattern': pattern,
                'SNR': snr,
                'sentence1': speaker1['sentence'],
                'sentence2': speaker2['sentence'],
                **audio_info  
            })
            
        except Exception as e:
            print(f"Ошибка в комбинации {combo_id}: {e}")
            continue

    return pd.DataFrame(result_data, columns=result_columns)

In [ ]:
# Генерация данных
result_df = create_audio_combinations(
    male_df=male_speakers,
    female_df=female_speakers,
    base_audio_dir=clips_dir,  
    output_dir="combinations",
    patterns=patterns,
    snr_levels=snr_levels,
    total_combinations=90
)

In [16]:
result_df.to_csv('audio_combinations.csv', index=False, encoding='utf-8-sig')

In [17]:
test = pd.read_csv('audio_combinations.csv')

In [18]:
test.head(10)

,ID,genders,pattern,SNR,file1,file2,output_file,sentence1,sentence2,duration,overlap_ratio
0,1,М-М,Частичное перекрытие,-5,common_voice_ru_18928687.mp3,common_voice_ru_37292770.mp3,combo_1.wav,Послышался злобный смех.,А он всё больше и больше хочет уйти от меня.,3.744,0.500000
1,2,М-М,Частичное перекрытие,0,common_voice_ru_35279768.mp3,common_voice_ru_21961235.mp3,combo_2.wav,Она села и начала расспрашивать Левина о его ж...,Он также настоятельно призвал соседние страны ...,4.860,0.500000
2,3,М-М,Частичное перекрытие,5,common_voice_ru_19593636.mp3,common_voice_ru_29263687.mp3,combo_3.wav,"Прилагая такие усилия, необходимо стараться уч...",Я вас узнать не могу.,9.096,0.288918
3,4,М-М,Параллельный диалог,-5,common_voice_ru_20792848.mp3,common_voice_ru_21889094.mp3,combo_4.wav,Поэтому наша делегация приветствует включение ...,Именно поэтому этот пункт сохраняется в повест...,7.296,0.796053
4,5,М-М,Параллельный диалог,0,common_voice_ru_21214070.mp3,common_voice_ru_20792810.mp3,combo_5.wav,Это поможет Африке твердо гарантировать свое б...,"Я потратил годы, защищая Израиль на полях сраж...",4.656,1.000000
5,6,М-М,Параллельный диалог,5,common_voice_ru_32349216.mp3,common_voice_ru_18891506.mp3,combo_6.wav,Степан Трофимович крепко держал ее за руку.,Ассамблея приступает к принятию решения по дан...,3.816,1.000000
6,7,М-Ж,Частичное перекрытие,-5,common_voice_ru_18991883.mp3,common_voice_ru_18932807.mp3,combo_7.wav,Я твоя сестра… Я старше.,"– Нюрка, – спросила ее бабка, – ты не видала, ...",3.624,0.500000
7,8,М-Ж,Частичное перекрытие,0,common_voice_ru_35228327.mp3,common_voice_ru_18940587.mp3,combo_8.wav,А лейкемия относится к раковым заболеваниям?,Во всех пунктах его приветствовала многочислен...,4.068,0.500000
8,9,М-Ж,Частичное перекрытие,5,common_voice_ru_37393940.mp3,common_voice_ru_26078720.mp3,combo_9.wav,"И рад, что был.",Мы должны предпринимать действия в отношении с...,2.988,0.500000
9,10,М-Ж,Параллельный диалог,-5,common_voice_ru_20719873.mp3,common_voice_ru_18913577.mp3,combo_10.wav,Выступление президента Украины господина Викто...,Я вот уже десять лет являюсь членом исполкома ...,6.096,1.000000
